In [125]:
### document datastructure

from langchain_core.documents import Document

In [126]:
doc=Document(
    page_content="This is a test document.",
    metadata={
        "source": "test_document.txt", 
        "pages":1,
        "author": "Shubh_Coder17", 
        "date": "2026-11-07"
    }
)
doc

Document(metadata={'source': 'test_document.txt', 'pages': 1, 'author': 'Shubh_Coder17', 'date': '2026-11-07'}, page_content='This is a test document.')

In [127]:
### Create a simple  txt file
import os
os.makedirs("../data/test_data", exist_ok=True)

In [128]:
sample_texts = {
    "../data/test_data/test_document.txt": """This is a test document.
    lorem ipsum dolor sit amet, consectetur adipiscing elit. Sed do eiusmod tempor incididunt ut labore et dolore magna aliqua. Ut enim ad minim veniam, quis nostrud exercitation ullamco laboris nisi ut aliquip ex ea commodo consequat. Duis aute irure dolor in reprehenderit in voluptate velit esse cillum dolore eu fugiat nulla pariatur. Excepteur sint occaecat cupidatat non proident, sunt in culpa qui officia deserunt mollit anim id est laborum.
    lorem ipsum dolor sit amet, consectetur adipiscing elit. Sed do eiusmod tempor incididunt ut labore et dolore magna aliqua. Ut enim ad minim veniam, quis nostrud exercitation ullamco laboris nisi ut aliquip ex ea commodo consequat. Duis aute irure dolor in reprehenderit in voluptate velit esse cillum dolore eu fugiat nulla pariatur. Excepteur sint occaecat cupidatat non proident, sunt in culpa qui officia deserunt mollit anim id est laborum.
    lorem-ipsum.multipleParagraphs: Lorem ipsum dolor sit amet, consectetur adipiscing elit. Sed do eiusmod tempor incididunt ut labore et dolore magna aliqua. Ut enim ad minim veniam, quis nostrud exercitation ullamco laboris nisi ut aliquip ex ea commodo consequat. Duis aute irure dolor in reprehenderit in voluptate velit esse cillum dolore eu fugiat nulla pariatur. Excepteur sint occaecat cupidatat non proident, sunt in culpa qui officia deserunt mollit anim id est laborum."""
}
for filepath, content in sample_texts.items():
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(content)
print ("Sample text files created successfully.")


Sample text files created successfully.


In [129]:
### text Loader

from langchain_community.document_loaders import TextLoader

loader = TextLoader("../data/test_data/test_document.txt", encoding="utf-8")
document = loader.load()
print(document)

[Document(metadata={'source': '../data/test_data/test_document.txt'}, page_content='This is a test document.\n    lorem ipsum dolor sit amet, consectetur adipiscing elit. Sed do eiusmod tempor incididunt ut labore et dolore magna aliqua. Ut enim ad minim veniam, quis nostrud exercitation ullamco laboris nisi ut aliquip ex ea commodo consequat. Duis aute irure dolor in reprehenderit in voluptate velit esse cillum dolore eu fugiat nulla pariatur. Excepteur sint occaecat cupidatat non proident, sunt in culpa qui officia deserunt mollit anim id est laborum.\n    lorem ipsum dolor sit amet, consectetur adipiscing elit. Sed do eiusmod tempor incididunt ut labore et dolore magna aliqua. Ut enim ad minim veniam, quis nostrud exercitation ullamco laboris nisi ut aliquip ex ea commodo consequat. Duis aute irure dolor in reprehenderit in voluptate velit esse cillum dolore eu fugiat nulla pariatur. Excepteur sint occaecat cupidatat non proident, sunt in culpa qui officia deserunt mollit anim id es

In [130]:
### Directory Loader
from langchain_community.document_loaders import DirectoryLoader

## Load all text files from a directory
directory_loader = DirectoryLoader(
 "../data/test_data", 
glob="**/*.txt", 
loader_cls=TextLoader, 
loader_kwargs={"encoding": "utf-8"},
show_progress = False
)
documents = directory_loader.load()
documents

[Document(metadata={'source': '..\\data\\test_data\\test_document.txt'}, page_content='This is a test document.\n    lorem ipsum dolor sit amet, consectetur adipiscing elit. Sed do eiusmod tempor incididunt ut labore et dolore magna aliqua. Ut enim ad minim veniam, quis nostrud exercitation ullamco laboris nisi ut aliquip ex ea commodo consequat. Duis aute irure dolor in reprehenderit in voluptate velit esse cillum dolore eu fugiat nulla pariatur. Excepteur sint occaecat cupidatat non proident, sunt in culpa qui officia deserunt mollit anim id est laborum.\n    lorem ipsum dolor sit amet, consectetur adipiscing elit. Sed do eiusmod tempor incididunt ut labore et dolore magna aliqua. Ut enim ad minim veniam, quis nostrud exercitation ullamco laboris nisi ut aliquip ex ea commodo consequat. Duis aute irure dolor in reprehenderit in voluptate velit esse cillum dolore eu fugiat nulla pariatur. Excepteur sint occaecat cupidatat non proident, sunt in culpa qui officia deserunt mollit anim id

In [131]:
### Directory Loader
from langchain_community import vectorstores
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader

from rag.main import EmbeddingManager

## Load all text files from a directory
directory_loader = DirectoryLoader(
 "../data/pdf", 
glob="**/*.pdf", 
loader_cls=PyMuPDFLoader, 
###loader_kwargs={"encoding": "utf-8"},
show_progress = False
)
pdf_documents = directory_loader.load()
pdf_documents

from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
from rag.main import VectorStore

# 1. Split the PDF documents into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
pdf_chunks = text_splitter.split_documents(pdf_documents)
print(f"Split {len(pdf_documents)} PDF pages into {len(pdf_chunks)} chunks.")

# 2. Generate embeddings for the PDF chunks
pdf_texts = [doc.page_content for doc in pdf_chunks]
# Use the notebook's shared embedding manager if it already exists.
if "embedding_manager" not in globals():

    class EmbeddingManager:
        def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
            self.model_name = model_name
            self.model = SentenceTransformer(self.model_name)

        def generate_embedding(self, text):
            if isinstance(text, str):
                text = [text]
            return self.model.encode(text, show_progress_bar=False)

    embedding_manager = EmbeddingManager()

pdf_embeddings = embedding_manager.generate_embedding(pdf_texts)

# 3. Add them to the vector store
# 3. Add them to the vector store
# Reuse the notebook's shared vectorstore if it already exists; otherwise create one here.
if "vectorstore" not in globals():
    vectorstore = VectorStore(collection_name="pdf_documents", persist_directory="../data/vector_store")

if len(pdf_chunks) != len(pdf_embeddings):
    raise ValueError(
        f"Chunk count ({len(pdf_chunks)}) must match embedding count ({len(pdf_embeddings)})."
    )

vectorstore.add_documents(pdf_chunks, pdf_embeddings)


Split 52 PDF pages into 284 chunks.
Generating embedding...


Batches: 100%|██████████| 9/9 [00:10<00:00,  1.16s/it]


Adding 284 documents to vector store...
Successfully added 284 documents to vector store.
Total documents in collection: 2290


In [132]:
type(
    pdf_documents[0]
)

langchain_core.documents.base.Document

### Embedding and VectorStoreDB

In [133]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [134]:
class EmbeddingManager:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model()
    
    def _load_model(self):
        try:
            print(f"Loading model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model {self.model_name} loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model: {e}")
            raise

    def generate_embedding(self, text) -> np.ndarray:
        if not self.model:
            raise ValueError("Model is not loaded. Call _load_model() first.")
        
        # Always convert single text into a list
        if isinstance(text, str):
            text = [text]
            
        print(f"Generating embedding...")
        embedding = self.model.encode(text, show_progress_bar=True)
        return embedding

# Initialize the embedding manager outside the class definition
embedding_manager = EmbeddingManager()
embedding_manager


Loading model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1492.72it/s]


Model all-MiniLM-L6-v2 loaded successfully. Embedding dimension: 384


C:\Users\1705p\AppData\Local\Temp\ipykernel_29064\1054640040.py:11: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model {self.model_name} loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


### VectorStore

In [135]:
from typing import List, Any
import os
import chromadb
import uuid
import numpy as np

class VectorStore:
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        try:
            # create a persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            #Get or create the collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "Collection of PDF document embeddings for RAG"}
            )
            print(f"VectorStore initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        if len(documents) != len(embeddings):
            raise ValueError("Number of Documents must match number of embeddings")
        print(f"Adding {len(documents)} documents to vector store...")

        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # prepare metadata and document text
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            # Document content
            documents_text.append(doc.page_content)

            # Embedding
            embeddings_list.append(embedding.tolist())

        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                metadatas=metadatas,
                documents=documents_text,
                embeddings=embeddings_list
            )
            print(f"Successfully added {len(documents)} documents to vector store.")
            print(f"Total documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore = VectorStore()
vectorstore

VectorStore initialized. Collection: pdf_documents
Existing documents in collection: 2290


In [136]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Initialize the text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

# 2. Split your documents into chunks
chunks = text_splitter.split_documents(documents)  # or pdf_documents

# 3. View the chunks
chunks

[Document(metadata={'source': '..\\data\\test_data\\test_document.txt'}, page_content='This is a test document.\n    lorem ipsum dolor sit amet, consectetur adipiscing elit. Sed do eiusmod tempor incididunt ut labore et dolore magna aliqua. Ut enim ad minim veniam, quis nostrud exercitation ullamco laboris nisi ut aliquip ex ea commodo consequat. Duis aute irure dolor in reprehenderit in voluptate velit esse cillum dolore eu fugiat nulla pariatur. Excepteur sint occaecat cupidatat non proident, sunt in culpa qui officia deserunt mollit anim id est laborum.'),
 Document(metadata={'source': '..\\data\\test_data\\test_document.txt'}, page_content='lorem ipsum dolor sit amet, consectetur adipiscing elit. Sed do eiusmod tempor incididunt ut labore et dolore magna aliqua. Ut enim ad minim veniam, quis nostrud exercitation ullamco laboris nisi ut aliquip ex ea commodo consequat. Duis aute irure dolor in reprehenderit in voluptate velit esse cillum dolore eu fugiat nulla pariatur. Excepteur si

### Convert the text to embeddings

In [137]:
texts=[doc.page_content for doc in chunks]

embeddings= embedding_manager.generate_embedding(texts)

vectorstore.add_documents(chunks, embeddings)

Generating embedding...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00,  7.46it/s]

Adding 3 documents to vector store...


Successfully added 3 documents to vector store.
Total documents in collection: 2293


### Retriever Pipeline From VectorStore

In [138]:
class RagRetriever:
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top k: {top_k}, Score Threshold: {score_threshold}")

        query_embedding = self.embedding_manager.generate_embedding([query])[0]

        try:
            results = self.vector_store.collection.query(
                query_embeddings=query_embedding.tolist(),
                n_results=top_k
            )
            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    score = 1 / (1 + distance) # Convert distance to similarity score
                    if score >= score_threshold:
                        retrieved_docs.append({
                            "id": doc_id,
                            "document": document,
                            "metadata": metadata,
                            "score": score
                        })
                print(f"Retrieved {len(retrieved_docs)} documents above the score threshold of {score_threshold}.")
            else:
                print("No documents retrieved.")
            return retrieved_docs
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever = RagRetriever(vectorstore, embedding_manager)

In [139]:
RagRetriever

__main__.RagRetriever

In [140]:
rag_retriever.retrieve("what is seating layout?")

Retrieving documents for query: 'what is seating layout?'
Top k: 5, Score Threshold: 0.0
Generating embedding...


Batches: 100%|██████████| 1/1 [00:00<00:00, 23.81it/s]

Retrieved 5 documents above the score threshold of 0.0.


[{'id': 'doc_0fb9103b_146',
  'document': "spacious rear-seat accommodation are usually considered\nsports sedans rather than sports cars.\nThe 1993–1998 McLaren F1 is notable for using a three-seat layout, where the front row consists of a\ncentrally located driver's seat.\nThe location of the engine and driven wheels significantly influence the handling characteristics of a car\nand are therefore crucial in the design of a sports car.[28][29][30][31][32] Traditionally, most sports cars have",
  'metadata': {'moddate': '2026-11-09T06:03:20+00:00',
   'title': 'Sports car - Wikipedia',
   'keywords': '',
   'creator': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) HeadlessChrome/154.0.0.0 Safari/537.36',
   'creationDate': "D:20261109060320+00'00'",
   'doc_index': 146,
   'modDate': "D:20261109060320+00'00'",
   'trapped': '',
   'source': '..\\data\\pdf\\Sports_car.pdf',
   'file_path': '..\\data\\pdf\\Sports_car.pdf',
   'format': 'PDF 1.4',
   'total_pages'

### Integration Vectordb Context pipeline With LLM output

In [141]:
### simple RAG-pipeline with Groq LLM
from langchain_core.messages import HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
import os
load_dotenv()

api_key = os.getenv("GOOGLE_API_KEY")

if not api_key:
    raise ValueError("GOOGLE_API_KEY not found in .env file")

### Initialize the Gemini LLM
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.1,
    max_output_tokens=1024,
    google_api_key=os.getenv("GOOGLE_API_KEY")
)


## simple RAG function: retrieve context + generate response
def rag_simple(query, retriever, llm, top_k = 5):
    ## retriever the context
    results = retriever.retrieve(query, top_k=top_k)
    context = "\n\n".join([doc ['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant documents found."

    ## generate response with context
    prompt = f"""Use the following context to answer the question concisely.
    Context: {context}
    Question: {query}
    Answer: """

    response = llm.invoke([prompt.format(context=context, query=query)])
    return response.content


e:\RAG\.venv\Lib\site-packages\google\genai\types.py:18161: SyntaxWarning: invalid escape sequence '\/'
  description="""Output only. The original name of the uploaded file, taken from the `Content-Disposition` header on upload. Between 1 and 255 characters. Control characters and the characters `<>:"|?*\/` are not accepted.""",
e:\RAG\.venv\Lib\site-packages\google\genai\types.py:18199: SyntaxWarning: invalid escape sequence '\/'
  """Output only. The original name of the uploaded file, taken from the `Content-Disposition` header on upload. Between 1 and 255 characters. Control characters and the characters `<>:"|?*\/` are not accepted."""


### Enhanced RAG Pipeline Features

In [142]:
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.3, return_context=False):
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant documents found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    context = "\n\n".join([doc['document'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'Unknown')),
        'page': doc['metadata'].get('page', 'Unknown'),
        'score': doc['score'],
        'preview': doc['document'][:300] + '...'
    } for doc in results]
    confidence = max([doc['score'] for doc in results])

    ## Generate Answer
        # Build the prompt
    final_prompt = f"Use the following context to answer the question: {query}\n\nContext: {context}"
    
    # Send as a list of messages (Groq requires this format to avoid 'Bad Request')
    response = llm.invoke([("human", final_prompt)])

    answer = response.content if hasattr(response, 'content') else str(response)

    output = {
        'answer': answer,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example Usage

result = rag_advanced(
    "What is the seating layout?",
    rag_retriever,
    llm,
    top_k=5,
    min_score=0.3,
    return_context=True
)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'What is the seating layout?'
Top k: 5, Score Threshold: 0.3
Generating embedding...


Batches: 100%|██████████| 1/1 [00:00<00:00, 35.72it/s]

Retrieved 5 documents above the score threshold of 0.3.


GoogleModelNotFoundError: Error calling model 'gemini-2.5-flash' (NOT_FOUND): 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use models/gemini-3.8-flash for the latest features and improvements. We recommend you to use the Interactions API (https://aistudio.google.com/docs/get-started).', 'status': 'NOT_FOUND'}}